In [ ]:
import os
import time
import numpy as np
import cv2
import torch
import torch.nn.functional as F
import networkx as nx
import matplotlib.pyplot as plt

from ground_truth.dataloader import MogeGtLoader
from ground_truth.feature_extraction import hook_df, hook_angle
from ground_truth.visualization import plot_lines_bool, plot_coplanar_lines
from ground_truth.utility_methods import (
    sobel_line, find_line_planes, get_line_pixels_trim,
    ransac_plane_fit, compute_distance_to_plane
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def load_and_resize_color(image_dir,image_id):
    image_dir = os.path.join("data", image_id)
    moge_gt_loader = MogeGtLoader(os.path.join("data", image_id))
    color_img = moge_gt_loader.load_color_image()
    h, w = color_img.shape[:2]
    return cv2.resize(color_img, (w//2, h//2), interpolation=cv2.INTER_AREA)


def reproject_depth_to_points_gpu(depth, K):
    """
    Reproject depth map to 3D world points on GPU.
    depth: [B×1×H×W], K: [B×3×3]
    returns: [B×3×H×W]
    """
    B, C, H, W = depth.shape
    # pixel coordinates
    i = torch.arange(W, device=device).view(1,1,W).expand(B,H,W)
    j = torch.arange(H, device=device).view(1,H,1).expand(B,H,W)
    ones = torch.ones_like(i)
    pixels = torch.stack([i, j, ones], dim=1).float()   # B×3×H×W
    K_inv = torch.inverse(K)
    dirs = K_inv.unsqueeze(-1).unsqueeze(-1) @ pixels.unsqueeze(-1)
    dirs = dirs.squeeze(-1)                           # B×3×H×W
    pts = dirs * depth                                # scale by depth
    return pts


def compute_normals_gpu(world_pts):
    """
    Compute normals via finite diff (Sobel) cross-product on GPU.
    world_pts: [B×3×H×W]
    returns: [B×3×H×W]
    """
    # Sobel kernels
    sobel = torch.tensor(
        [[1,0,-1],[2,0,-2],[1,0,-1]],
        dtype=world_pts.dtype,
        device=device
    ).view(1,1,3,3)
    # gradient per channel
    gx = F.conv2d(world_pts, sobel, padding=1, groups=3)
    gy = F.conv2d(world_pts, sobel.transpose(2,3), padding=1, groups=3)
    # cross product
    normals = torch.cross(gx, gy, dim=1)
    norm = torch.linalg.norm(normals, dim=1, keepdim=True)
    normals = normals / (norm + 1e-8)
    return normals


def threshold_and_morph(tensor, thr):
    """
    Threshold > thr then morphological close (erode→dilate) on GPU.
    Input: [B×1×H×W], returns same shape binary float tensor.
    """
    mask = (tensor > thr).float()
    kernel = torch.ones(1,1,3,3, device=device)
    # erode: only keep where all 9 neighbors are 1
    eroded = (F.conv2d(mask, kernel, padding=1) == 9).float()
    # dilate: any neighbor 1
    closed = (F.conv2d(eroded, kernel, padding=1) >= 1).float()
    return closed


def process_image(
    image_dir, image_id, frame_str, net, moge_model,
    depth_thr=0.1, normal_thr=9e13,
    thickness=1,
    text_color=(255,0,0), non_struct_color=(128,128,128)
):
    start = time.time()

    # 1) Load & resize
    img_path = os.path.join(image_dir, image_id)
    color = load_and_resize_color(img_path,image_id)
    h, w = color.shape[:2]

    # 2) Moge inference (GPU)
    inp = torch.from_numpy(color/255.0).permute(2,0,1).unsqueeze(0).to(device)
    out = moge_model.infer(inp)
    depth = out["depth"]           # [B×1×H×W]
    K     = out["intrinsics"]      # [B×3×3]

    # 3) Reproject & normals on GPU
    world_pts = reproject_depth_to_points_gpu(depth, K)
    normals   = compute_normals_gpu(world_pts)

    # 4) Edge detection on GPU
    # Sobel on depth
    sobel_kx = torch.tensor(
        [[1,0,-1],[2,0,-2],[1,0,-1]],
        dtype=depth.dtype,
        device=device
    ).view(1,1,3,3)
    sobel_ky = sobel_kx.transpose(2,3)
    dx = F.conv2d(depth, sobel_kx, padding=1)
    dy = F.conv2d(depth, sobel_ky, padding=1)
    sobel_depth = torch.sqrt(dx**2 + dy**2)
    # Laplace on normals
    lap = torch.tensor(
        [[0,1,0],[1,-4,1],[0,1,0]],
        dtype=depth.dtype,
        device=device
    ).view(1,1,3,3)
    nx_ = F.conv2d(normals[:,0:1], lap, padding=1)
    ny_ = F.conv2d(normals[:,1:2], lap, padding=1)
    nz_ = F.conv2d(normals[:,2:3], lap, padding=1)
    sobel_normal = torch.sqrt(nx_**2 + ny_**2 + nz_**2)

    # Threshold + morphological close
    mask_d = threshold_and_morph(sobel_depth, depth_thr)
    mask_n = threshold_and_morph(sobel_normal, normal_thr)
    combined = mask_d | mask_n

    # 5) Transfer once to CPU
    mask_cpu = (combined[0,0].cpu().numpy().astype(np.uint8) * 255)
    world_cpu = world_pts[0].permute(1,2,0).cpu().numpy()

    # 6) Connected Components & RANSAC on CPU
    num_labels, labels_im = cv2.connectedComponents(mask_cpu)
    cluster_planes = {}
    for lbl in range(1, num_labels):
        pts = world_cpu[labels_im == lbl]
        if pts.shape[0] < 50:
            continue
        model, inliers = ransac_plane_fit(pts, num_iterations=50, threshold=0.03, min_inliers_ratio=0.8)
        if model is None:
            continue
        in_pts = pts[inliers]
        errs = compute_distance_to_plane(in_pts, model[0], model[1])
        if errs.mean() > 0.05:
            continue
        # Least-squares refine
        A = np.c_[in_pts[:,0], in_pts[:,1], np.ones(in_pts.shape[0])]
        sol, _, _, _ = np.linalg.lstsq(A, in_pts[:,2], rcond=None)
        n_ls = np.array([sol[0], sol[1], -1])
        n_ls = n_ls / np.linalg.norm(n_ls)
        d_ls = sol[2]
        cluster_planes[lbl] = {'ls': (n_ls, d_ls)}

    # 7) Merge via Region Adjacency Graph (NetworkX)
    valid_labels = list(cluster_planes.keys())
    masks = {}
    dilates = {}
    kernel = np.ones((16,16), np.uint8)
    for lbl in valid_labels:
        m = (labels_im == lbl).astype(np.uint8)
        masks[lbl] = m
        dilates[lbl] = cv2.dilate(m, kernel, iterations=7)

    G = nx.Graph()
    for lbl in valid_labels:
        G.add_node(lbl)
    for i, l1 in enumerate(valid_labels):
        for l2 in valid_labels[i+1:]:
            if (masks[l2] is not None and masks[l1] is not None
               and np.any(cv2.bitwise_and(dilates[l1], masks[l2]))):
                n1, d1 = cluster_planes[l1]['ls']
                n2, d2 = cluster_planes[l2]['ls']
                if np.abs(np.dot(n1, n2)) > 0.95 and np.abs(d1 - d2) < 0.04:
                    G.add_edge(l1, l2)
    merged_groups = list(nx.connected_components(G))
    merged_map = np.zeros_like(labels_im)
    new_lbl = 1
    for grp in merged_groups:
        for lbl in grp:
            merged_map[labels_im == lbl] = new_lbl
        new_lbl += 1

    # 8) DeepLSD line detection on GPU
    gray = cv2.cvtColor(color, cv2.COLOR_RGB2GRAY)
    inp_lsd = torch.from_numpy(gray[None,None]/255.0).to(device)
    df_handle = net.df_head[5].register_forward_hook(hook_df)
    ang_handle = net.angle_head[5].register_forward_hook(hook_angle)
    with torch.no_grad():
        out2 = net({'image': inp_lsd})
    pred_lines = out2['lines'][0].cpu().numpy()
    df_handle.remove()
    ang_handle.remove()

    # 9) Classify structural vs textural
    is_struct = []
    is_depth_sep = []
    for l in pred_lines:
        md, mn = sobel_line(mask_cpu, mask_cpu, l)
        on_d = np.any(md)
        on_n = np.any(mn)
        is_struct.append(on_d or on_n)
        is_depth_sep.append(on_d)

    # Draw lines on copy
    composite = color.copy()
    for i, l in enumerate(pred_lines):
        pts = l.reshape(2,2)
        col = non_struct_color if not is_struct[i] else text_color
        cv2.line(composite,
                 (int(round(pts[0,0])), int(round(pts[0,1]))),
                 (int(round(pts[1,0])), int(round(pts[1,1]))),
                 col, thickness)

    # 10) Final plotting
    plt.figure(figsize=(20,25))
    axs = [plt.subplot(4,2,i+1) for i in range(8)]
    titles = [
        'Depth Map', 'Normal Map', 'Sobel Depth', 'Sobel Normal',
        'Thresholded Depth+Normal', 'Mask', 'Lines Overlay', 'Original'
    ]
    imgs = [
        depth[0,0].cpu().numpy(),
        normals[0].permute(1,2,0).cpu().numpy(),
        sobel_depth[0,0].cpu().numpy(),
        sobel_normal[0,0].cpu().numpy(),
        (mask_cpu),
        (mask_cpu),
        composite,
        cv2.cvtColor(color, cv2.COLOR_BGR2RGB)
    ]
    cmaps = ['gray','gray','gray','gray','gray','gray',None,None]
    for ax, im, ti, cm in zip(axs, imgs, titles, cmaps):
        ax.set_title(ti)
        ax.axis('off')
        ax.imshow(im, cmap=cm)
    plt.show()

    print(f"Total runtime: {time.time() - start:.3f} s")


In [6]:
frame_str = "0001"
desired_images = [
    "ai_001_001",
    "ai_001_004",
    "ai_001_005",
    "ai_001_006",
    "ai_001_007",
    "ai_001_008",
    "ai_001_009",
    "ai_002_001",

    #"DSC_0442",
    #"DSC_0455",
    #"DSC_0239",
    #"DSC_0249",
]

desired_images = [
    #"ai_001_001",
    #"ai_001_004",
    #"ai_001_005",
    #"ai_001_006",
    #"ai_001_007",
    #"ai_001_008",
    #"ai_001_009",
    #"ai_002_001",

   
    "DSC_0239",
    "DSC_0249",
    "DSC_0298",
    "DSC_0299",
    "DSC_0300",
    "DSC_0340",
    "DSC_0341",
    "DSC_0342",
    "DSC_0442",
    "DSC_0455",
]
from moge.model.v1 import MoGeModel
import os
import torch
from deeplsd.models.deeplsd_inference import DeepLSD

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
conf = {'detect_lines': True, 'line_detection_params': {'merge': False, 'filtering': True, 'grad_thresh': 3}}
ckpt = torch.load('../weights/deeplsd_md.tar', map_location='cpu', weights_only=False)
net = DeepLSD(conf)
net.load_state_dict(ckpt['model'])
net = net.to(device).eval()


moge = MoGeModel.from_pretrained("Ruicheng/moge-vitl").to(device)



# Process, plot, and save JSON data for each image.
for image_id in desired_images:
    #composite_after, pred_lines, img, normals, world_coordinates, valid_mask, line_info, scores, isstruct, original_lines
    # Load the model from huggingface hub (or load from local).
    image_dir = os.path.join("data", image_id)
    _ = process_image(
    image_dir, image_id, frame_str, net, moge,
    depth_thr=0.1, normal_thr=9e13,
    thickness=1,
    text_color=(255,0,0), non_struct_color=(128,128,128)
    )
    

/home/shangeeth/wsl_deeplsd_39/lib/python3.9/site-packages/moge/model/v1.py:171: UserWarning: The following deprecated/invalid arguments are ignored: {'output_mask': True, 'split_head': True}
  warnings.warn(f"The following deprecated/invalid arguments are ignored: {deprecated_kwargs}")
INFO: using MLP layer as FFN


AttributeError: 'NoneType' object has no attribute 'shape'